In [248]:
# import
import pandas as pd
import os
import numpy as np

In [249]:
def safe_read_csv(filepath: str) -> pd.DataFrame:
    """
    여러 인코딩을 시도하여 CSV 파일을 안전하게 읽는 함수

    Parameters:
        filepath (str): CSV 파일 경로

    Returns:
        pd.DataFrame: 성공적으로 로드된 DataFrame
    """
    # 사용할 인코딩 후보 리스트
    encodings = ["cp949", "utf-8-sig", "utf-8", "euc-kr"]
    last_error = None  # 마지막으로 발생한 오류 저장

    # 후보 인코딩을 순서대로 시도
    for enc in encodings:
        try:
            # 주어진 인코딩으로 CSV 읽기
            df = pd.read_csv(filepath, encoding=enc)

            # 읽은 파일이 비어 있으면 오류 발생
            if df.empty:
                raise ValueError("CSV 파일이 비어 있습니다.")

            # 정상적으로 읽었으면 DataFrame 반환
            return df

        except Exception as e:
            # 실패하면 오류 기록 후 다음 인코딩 시도
            last_error = e
            continue

    # 모든 인코딩 시도 후에도 실패한 경우 예외 발생
    raise ValueError(f"CSV 로드 실패: {last_error}")


In [250]:
# 시도와 시군구 컬럼을 결합하여 하나의 'SGG_NAME' 컬럼을 생성
def combine_region_columns(
    df: pd.DataFrame,
    col_sido: str = "시군구별(1)",
    col_sigungu: str = "시군구별(2)",
    new_col: str = "시군구별"
) -> pd.DataFrame:
    """
    시도와 시군구 컬럼을 결합하여 하나의 'SGG_NAME' 컬럼을 생성

    Parameters:
        df (pd.DataFrame): 원본 DataFrame
        col_sido (str): 시도 컬럼명 (기본값: '시군구별(1)')
        col_sigungu (str): 시군구 컬럼명 (기본값: '시군구별(2)')
        new_col (str): 생성될 컬럼명 (기본값: '시군구_전체')

    Returns:
        pd.DataFrame: 시군구_전체 컬럼이 추가된 DataFrame
    """
    # 0. 소계 제거 (단, 세종특별자치시 소계는 보존)
    df = df[~((df["시군구별(2)"] == "소계") & (df["시군구별(1)"] != "세종특별자치시"))]
    # df = df[~((df["시군구별(2)"] == "소계"))]
    
    # 1. 시군구 결합
    df[new_col] = df[col_sido].str.strip() + " " + df[col_sigungu].str.strip()

    # 2. 세종특별자치시 소계 → 세종특별자치시 로 치환
    df.loc[df[new_col] == "세종특별자치시 소계", new_col] = "세종특별자치시"

    # 3. 기존 컬럼 제거
    df.drop(columns=[col_sido, col_sigungu], inplace=True)

    # 4. 새 컬럼을 가장 왼쪽으로 이동
    cols = [new_col] + [col for col in df.columns if col != new_col]
    df = df[cols]


    return df


In [251]:
# 시/도 이름 정규화
def normalize_sido_names(
    df: pd.DataFrame, 
    column: str, 
    col_cnt: int
) -> pd.DataFrame:
    """
    주어진 컬럼에서 시도 약칭을 공통된 명칭으로 표준화, 시군구 컬럼명 통일('SGG_NAME')

    Parameters:
        df (pd.DataFrame): 대상 DataFrame
        column (str): 변환할 컬럼명 ("시군구별(1)" 또는 "시군구별")
        col_cnt (int): 시군구 컬럼 개수 (1 또는 2)

    Returns:
        pd.DataFrame: 치환된 DataFrame
    """
    
    # 1) 시도 약칭/옛 표기 → 정식 명칭 매핑 사전
    mapping = {
        "서울": "서울특별시", "인천": "인천광역시", "경기": "경기도",
        "부산": "부산광역시", "대구": "대구광역시", "광주": "광주광역시",
        "대전": "대전광역시", "울산": "울산광역시", "세종": "세종특별자치시",
        "강원": "강원특별자치도", "충북": "충청북도", "충남": "충청남도",
        "전북": "전북특별자치도", "전남": "전라남도", "경북": "경상북도",
        "경남": "경상남도", "제주": "제주특별자치도",
        "전라북도": "전북특별자치도", "강원도": "강원특별자치도"
}

    # 2) 대상 컬럼 존재 확인
    if column not in df.columns:
        raise KeyError(f"'{column}' 컬럼없음")

    # 3) 개별 값 변환 함수 정의
    def convert_region_name(value):
        value = str(value).strip()
        for short, full in mapping.items():
            # 3-1) 이미 정식 명칭이면 그대로 반환
            if value.startswith(full):
                return value
            # 3-2) "강원도", "충북도" 같이 도 단위 매칭
            if value.startswith(short + "도"):
                return value.replace(short + "도", full, 1)
            # 3-3) 약칭일 때 정식 명칭으로 치환
            if value.startswith(short):
                return f"{full} {value[len(short):].strip()}"
        return value

    # 4) 시도명 변환 실행 & 컬럼명 'SGG_NAME'으로 변경
    if col_cnt == 1:
        df[column] = df[column].apply(convert_region_name)
        df.rename(columns={column: "SGG_NAME"}, inplace=True)
    else:
        df[column] = df[column].apply(convert_region_name)
        df.rename(columns={"시군구별": "SGG_NAME"}, inplace=True)
    
    # 5) 예외 처리: 잘못된 표기 수정
    df["SGG_NAME"] = df["SGG_NAME"].replace({
        "세종특별자치시 세종시": "세종특별자치시",
        "세종특별자치시 세종특별자치시": "세종특별자치시",
        "경상남도 창원시(통합)": "경상남도 창원시",
        "부산광역시 진구": "부산진구",
        "제주특별자치도 제주특별자치도 시": "제주특별자치도 제주시",
    })
    
    # 6) 명백히 잘못된 중복 표기 제거
    df = df[df['SGG_NAME'] != '광주광역시 광주광역시']
    df = df[df['SGG_NAME'] != '부산광역시 부산광역시']

    # 7) 필요 시 중복 처리/정렬 (현재는 주석 처리)
    # df = df.groupby("SGG_NAME", as_index=False).first()
    # df = df.sort_values(by="SGG_NAME").reset_index(drop=True)
    
    return df

In [252]:
def merge_to_city(df: pd.DataFrame, region_col: str = "SGG_NAME") -> pd.DataFrame:
    """
    특례시 하위 구 단위 데이터를 시 단위로 병합해 새로운 행 생성해주는 함수수

    Parameters:
        df (pd.DataFrame): 원본 데이터프레임
        region_col (str): 시군구 전체 이름 컬럼명 (예: "시군구(전체)", "SGG_NAME" 등)

    Returns:
        pd.DataFrame: 병합 후 새로운 행 추가 및 기존 구 단위 행 제거된 DataFrame
    """
    merge_targets = {
    "경기도 수원시": [
        "경기도 수원시 장안구", "경기도 수원시 권선구", "경기도 수원시 팔달구", "경기도 수원시 영통구",
        "경기도 수원시장안구", "경기도 수원시권선구", "경기도 수원시팔달구", "경기도 수원시영통구",
        
    ],
    "경기도 고양시": [
        "경기도 고양시 덕양구", "경기도 고양시 일산동구", "경기도 고양시 일산서구",
        "경기도 고양시덕양구", "경기도 고양시일산동구", "경기도 고양시일산서구"
    ],
    "경기도 용인시": [
        "경기도 용인시 처인구", "경기도 용인시 기흥구", "경기도 용인시 수지구",
        "경기도 용인시처인구", "경기도 용인시기흥구", "경기도 용인시수지구"
    ],
    "경기도 성남시": [
        "경기도 성남시 수정구", "경기도 성남시 중원구", "경기도 성남시 분당구",
        "경기도 성남시수정구", "경기도 성남시중원구", "경기도 성남시분당구"
    ],
    "경기도 안산시": [
        "경기도 안산시 상록구", "경기도 안산시 단원구",
        "경기도 안산시상록구", "경기도 안산시단원구"
    ],
    "경기도 안양시": [
        "경기도 안양시 만안구", "경기도 안양시 동안구",
        "경기도 안양시만안구", "경기도 안양시동안구"
    ],
    "충청북도 청주시": [
        "충청북도 청주시 상당구", "충청북도 청주시 서원구",
        "충청북도 청주시 흥덕구", "충청북도 청주시 청원구",
        "충청북도 청주시상당구", "충청북도 청주시서원구",
        "충청북도 청주시흥덕구", "충청북도 청주시청원구"
    ],
    "경상남도 창원시": [
        "경상남도 창원시 의창구", "경상남도 창원시 성산구",
        "경상남도 창원시 마산합포구", "경상남도 창원시 마산회원구",
        "경상남도 창원시 진해구",
        "경상남도 창원시의창구", "경상남도 창원시성산구",
        "경상남도 창원시마산합포구", "경상남도 창원시마산회원구",
        "경상남도 창원시진해구",
    ],
    "충청남도 천안시": [
        "충청남도 천안시 동남구", "충청남도 천안시 서북구",
        "충청남도 천안시동남구", "충청남도 천안시서북구"
    ],
    "전북특별자치도 전주시": [
        "전북특별자치도 전주시 완산구", "전북특별자치도 전주시 덕진구",
        "전북특별자치도 전주시완산구", "전북특별자치도 전주시덕진구"
    ],
    "전라북도 전주시": [
        "전라북도 전주시 완산구", "전라북도 전주시 덕진구",
        "전라북도 전주시완산구", "전라북도 전주시덕진구"
    ],
    "경상북도 포항시": [
        "경상북도 포항시 북구", "경상북도 포항시 남구",
        "경상북도 포항시북구", "경상북도 포항시남구"
    ]
}

    # 시 단위로 합치기
    for new_name, old_names in merge_targets.items():
        subset = df[df[region_col].isin(old_names)]
        if not subset.empty:
            # 구 단위 값 합산 후 시 단위 행 추가
            summed = subset.drop(columns=[region_col]).sum(numeric_only=True)
            new_row = pd.DataFrame([{region_col: new_name, **summed.to_dict()}])
            df = pd.concat([df, new_row], ignore_index=True)

        # 구 단위 행 제거
        df = df[~df[region_col].isin(old_names)]

    # 이름 기준 정렬
    df = df.sort_values(by=region_col).reset_index(drop=True)

    return df


In [253]:
def make_full_sgg_names(
    df: pd.DataFrame, 
    column: str = "SGG_NAME", 
    drop_original: bool = False
) -> pd.DataFrame:
    """
    시도와 시군구를 합쳐서 표준화된 지역명 컬럼'SGG_NAME'을 생성

    Parameters:
        df (pd.DataFrame): 원본 DataFrame
        column (str): 시도/시군구가 포함된 컬럼명
        drop_original (bool): 원본 column을 삭제할지 여부 (기본: False)

    Returns:
        pd.DataFrame: 'SGG_NAME' 컬럼이 포함된 DataFrame
    """
    # 1) 시도 이름 목록 정의
    cities = ["서울특별시", "인천광역시", "경기도", "강원도", "충청남도", 
        "충청북도", "전라북도", "전라남도", "전북특별자치도",
        "경상남도", "경상북도", "강원특별자치도", "제주특별자치도",
        "대전광역시", "대구광역시", "부산광역시", "울산광역시",
        "광주광역시", "세종특별자치시"]

    current_city = None
    new_names = []

    # 2) 각 행을 돌면서 시/시군구 결합
    for idx, region in enumerate(df[column]):
        region = str(region).strip()
        if region in cities:
            # 시도명이면 현재 시도 갱신
            current_city = region
            new_names.append(region)
        elif current_city is not None:
            # 시군구면 앞에 시도 붙여서 저장
            new_names.append(f"{current_city} {region}")
        else:
            # 시도 없이 시군구만 나오면 오류
            raise ValueError(f"[{idx}행] 시도 정보 없이 시군구 '{region}'가 나왔습니다.")

    # 3) 항상 새로운 'SGG_NAME' 덮어쓰기
    df["SGG_NAME"] = new_names

    # 4) 필요 시 원본 컬럼 삭제
    if drop_original:
        df.drop(columns=[column], inplace=True)

    # 5) 컬럼 순서: SGG_NAME → (원본 column) → 나머지
    first_col = ["SGG_NAME"]
    if not drop_original and column != "SGG_NAME":
        first_col.append(column)
    remaining_cols = [col for col in df.columns if col not in first_col]
    df = df[first_col + remaining_cols]
    
    # 5) 잘못된, 중복 표기 예외 처리
    df["SGG_NAME"] = df["SGG_NAME"].replace({
        "세종특별자치시 세종시": "세종특별자치시",
        "세종특별자치시 소계" : "세종특별자치시",
        "세종특별자치시 세종": "세종특별자치시",
        "경상남도 창원시(통합)": "경상남도 창원시",
        "부산광역시 진구": "부산광역시 부산진구",
        "제주특별자치도 제주특별자치도 시": "제주특별자치도 제주시",
    })   
    
    return df


In [254]:
# 군위군 행정구역 변경 반영
def merge_gunwigun(
    df: pd.DataFrame, 
    region_col: str = "SGG_NAME"
) -> pd.DataFrame:
    """
    '경상북도 군위군'과 '대구광역시 군위군'의 데이터를 병합하여
    결측치를 보완하고 '대구광역시 군위군'으로 통합
    '경상북도 군위군'은 삭제

    Parameters:
        df (pd.DataFrame): 원본 DataFrame
        region_col (str): 행정구역 컬럼명

    Returns:
        pd.DataFrame: 병합된 결과
    """
    # 결측치로 변환
    df.replace("-", np.nan, inplace=True)

    # 두 행 필터링
    gw = df[df[region_col] == "경상북도 군위군"]
    dg = df[df[region_col] == "대구광역시 군위군"]

    if gw.empty and dg.empty:
        return df  # 둘 다 없으면 그대로 반환

    # 병합 기준: 값 있는 쪽으로 결측 채움
    gw_values = gw.drop(columns=[region_col]).squeeze() if not gw.empty else pd.Series()
    dg_values = dg.drop(columns=[region_col]).squeeze() if not dg.empty else pd.Series()

    # 결측치 채우기: gw → dg로 채움
    merged_values = dg_values.combine_first(gw_values)

    # 병합된 새 행 생성
    new_row = pd.DataFrame([{region_col: "대구광역시 군위군", **merged_values.to_dict()}])

    # 기존 행 제거 + 병합된 행 추가
    df = df[~df[region_col].isin(["경상북도 군위군", "대구광역시 군위군"])]
    df = pd.concat([df, new_row], ignore_index=True)
    
    # 이름 기준 정렬
    df = df.sort_values(by=region_col).reset_index(drop=True)
    df.to_csv("./test.csv", encoding="utf-8-sig", index=False)

    # 결측치 처리
    return df

In [255]:
# 특례시 하위 행정구역 제거
def remove_subdistricts(
    df: pd.DataFrame, 
    region_col: str = "SGG_NAME"
) -> pd.DataFrame:
    """
    시군구 통합 대상에 포함된 하위 행정구역을 제거하고,
    세종특별자치시 단독 행은 유지하며, 나머지 시도 단독 행은 제거
    이후 시군구 기준 정렬

    Parameters:
        df (pd.DataFrame): 처리 대상 데이터프레임
        region_col (str): 행정구역 전체 컬럼명

    Returns:
        pd.DataFrame: 필터링 및 정렬된 결과 DataFrame
    """

    # 제거할 행정구역 리스트
    regions_to_remove = [
        # 수원시
        "경기도 장안구", "경기도 권선구", "경기도 팔달구", "경기도 영통구",
        "경기도 수원시 장안구", "경기도 수원시 권선구", "경기도 수원시 팔달구", "경기도 수원시 영통구",
        "경기도 수원시장안구", "경기도 수원시권선구", "경기도 수원시팔달구", "경기도 수원시영통구",

        # 고양시
        "경기도 덕양구", "경기도 일산동구", "경기도 일산서구",
        "경기도 고양시 덕양구", "경기도 고양시 일산동구", "경기도 고양시 일산서구",
        "경기도 고양시덕양구", "경기도 고양시일산동구", "경기도 고양시일산서구",
        "경기도 고양시 덕양구", "경기도 고양시 일산 동구", "경기도 고양시 일산 서구",

        # 용인시
        "경기도 처인구", "경기도 기흥구", "경기도 수지구",
        "경기도 용인시 처인구", "경기도 용인시 기흥구", "경기도 용인시 수지구",
        "경기도 용인시처인구", "경기도 용인시기흥구", "경기도 용인시수지구",

        # 성남시
        "경기도 수정구", "경기도 중원구", "경기도 분당구",
        "경기도 성남시 수정구", "경기도 성남시 중원구", "경기도 성남시 분당구",
        "경기도 성남시수정구", "경기도 성남시중원구", "경기도 성남시분당구",

        # 안산시
        "경기도 상록구", "경기도 단원구",
        "경기도 안산시 상록구", "경기도 안산시 단원구",
        "경기도 안산시상록구", "경기도 안산시단원구",

        # 안양시
        "경기도 만안구", "경기도 동안구",
        "경기도 안양시 만안구", "경기도 안양시 동안구",
        "경기도 안양시만안구", "경기도 안양시동안구",
        
        # 청주
        "충청북도 상당구", "충청북도 서원구", "충청북도 흥덕구", "충청북도 청원구",
        "충청북도 청주시 상당구", "충청북도 청주시 서원구", "충청북도 청주시 흥덕구", "충청북도 청주시 청원구",
        "충청북도 청주시상당구", "충청북도 청주시서원구", "충청북도 청주시흥덕구", "충청북도 청주시청원구",
        
        # 창원
        "경상남도 의창구", "경상남도 성산구", "경상남도 마산합포구", "경상남도 마산회원구", "경상남도 진해구",
        "경상남도 창원시 의창구", "경상남도 창원시 성산구", "경상남도 창원시 마산합포구", "경상남도 창원시 마산회원구", "경상남도 창원시 진해구",
        "경상남도 창원시의창구", "경상남도 창원시성산구", "경상남도 창원시마산합포구", "경상남도 창원시마산회원구", "경상남도 창원시진해구",
        
        # 충남
        "충청남도 동남구", "충청남도 서북구", "충청남도 천안시 동남구", "충청남도 천안시 서북구",
        "충청남도 천안시동남구", "충청남도 천안시서북구",
        
        # 전북
        "전북특별자치도 완산구", "전북특별자치도 덕진구", "전북특별자치도 전주시 완산구", "전북특별자치도 전주시 덕진구",
        "전라북도 완산구", "전라북도 덕진구", "전라북도 전주시 완산구", "전라북도 전주시 덕진구",
        "전북특별자치도 전주시완산구", "전북특별자치도 전주시덕진구", "전라북도 전주시완산구", "전라북도 전주시덕진구",
        
        # 경북
        "경상북도 북구", "경상북도 남구", "경상북도 포항시 북구", "경상북도 포항시 남구",
        "경상북도 포항시북구", "경상북도 포항시남구",
    ]


    # 특정 행정구역 제거
    df = df[~df[region_col].isin(regions_to_remove)]

    # 정렬 및 인덱스 초기화
    df = df.sort_values(by=region_col).reset_index(drop=True)
    # df.to_csv("./test.csv", encoding="utf-8-sig", index=False)
    return df


In [256]:
def remove_only_sido_rows(
    df: pd.DataFrame, 
    region_col: str = "SGG_NAME"
) -> pd.DataFrame:
    """
    시도 단독 행을 제거하고, '충청북도 충북' 등의 잘못된 지역명도 함께 제거

    Parameters:
        df (pd.DataFrame): 원본 데이터프레임
        region_col (str): 시군구 전체 이름 컬럼명

    Returns:
        pd.DataFrame: 시도 단독 및 중복 명칭 제거된 DataFrame
    """
    # '세종특별자치시'는 예외로 인정 (시군구 없이 단일 도시)
    df = df[
        (df[region_col].str.contains("세종특별자치시")) |
        (df[region_col].str.contains(" "))  # 공백 포함된 시군구명 (즉, 시도+시군구 구조)
    ]

    # 추가적으로 충청북도 충북, 충청남도 충남 등 시도명이 두 번 반복된 잘못된 명칭 제거
    invalid_names = [
        "서울특별시 서울",
        "인천광역시 인천",
        "경기도 경기",
        "충청북도 충북",
        "충청남도 충남",
        "전북특별자치도 전북",
        "경상북도 경북",
        "경상남도 경남",
        "강원특별자치도 강원", 
        "제주특별자치도 제주",
        "세종특별자치시 세종",
        "대전광역시 대전",
        "대구광역시 대구",
        "광주광역시 광주",
        "울산광역시 울산",
        "부산광역시 부산",
        "전라남도 전남"
    ]
    df = df[~df[region_col].isin(invalid_names)]
    df = df.sort_values(by=region_col).reset_index(drop=True)

    return df


In [257]:
def check_missing_regions(
    df: pd.DataFrame, 
    region_col: str = "SGG_NAME"
) -> pd.DataFrame:
    """
    지정된 시군구 리스트를 기준으로 CSV 파일 내 누락된 지역을 확인하고 저장 또는 출력하는 함수

    Parameters:
        csv_path (str): 확인할 CSV 파일 경로
        region_col (str): 지역명이 들어 있는 컬럼명 (예: 'SGG_NAME')
        region_list (list[str]): 기준이 되는 전체 시군구 리스트
        save_path (str): 누락된 항목을 저장할 경로 (지정하지 않으면 출력만 함)

    Returns:
        list[str]: 누락된 지역 목록
    """
    
    region_list = [
    "서울특별시 종로구", "서울특별시 중구", "서울특별시 용산구", "서울특별시 성동구",
    "서울특별시 광진구", "서울특별시 동대문구", "서울특별시 중랑구", "서울특별시 성북구",
    "서울특별시 강북구", "서울특별시 도봉구", "서울특별시 노원구", "서울특별시 은평구",
    "서울특별시 서대문구", "서울특별시 마포구", "서울특별시 양천구", "서울특별시 강서구",
    "서울특별시 구로구", "서울특별시 금천구", "서울특별시 영등포구", "서울특별시 동작구",
    "서울특별시 관악구", "서울특별시 서초구", "서울특별시 강남구", "서울특별시 송파구",
    "서울특별시 강동구",
    "경기도 가평군", "경기도 고양시", "경기도 과천시", "경기도 광명시", "경기도 광주시", 
    "경기도 구리시", "경기도 군포시", "경기도 김포시", "경기도 남양주시", "경기도 동두천시", 
    "경기도 부천시", "경기도 성남시", "경기도 수원시", "경기도 시흥시", "경기도 안산시",
    "경기도 안성시", "경기도 안양시", "경기도 양주시", "경기도 양평군", "경기도 여주시", 
    "경기도 연천군", "경기도 오산시", "경기도 용인시", "경기도 의왕시", "경기도 의정부시", 
    "경기도 이천시", "경기도 파주시", "경기도 평택시", "경기도 포천시", "경기도 하남시", 
    "경기도 화성시",
    "인천광역시 강화군", "인천광역시 옹진군", "인천광역시 계양구", "인천광역시 미추홀구",
    "인천광역시 남동구", "인천광역시 동구", "인천광역시 부평구", "인천광역시 서구",
    "인천광역시 연수구", "인천광역시 중구",
    "강원특별자치도 강릉시", "강원특별자치도 고성군", "강원특별자치도 동해시", "강원특별자치도 삼척시", "강원특별자치도 속초시",
    "강원특별자치도 양구군", "강원특별자치도 양양군", "강원특별자치도 영월군", "강원특별자치도 원주시", "강원특별자치도 인제군",
    "강원특별자치도 정선군", "강원특별자치도 철원군", "강원특별자치도 춘천시", "강원특별자치도 태백시", "강원특별자치도 평창군",
    "강원특별자치도 홍천군", "강원특별자치도 화천군", "강원특별자치도 횡성군", "경상남도 거제시", "경상남도 거창군",
    "경상남도 고성군", "경상남도 김해시", "경상남도 남해군", "경상남도 밀양시", "경상남도 사천시", "경상남도 산청군",
    "경상남도 양산시", "경상남도 의령군", "경상남도 진주시", "경상남도 창녕군", "경상남도 창원시", "경상남도 통영시",
    "경상남도 하동군", "경상남도 함안군", "경상남도 함양군", "경상남도 합천군", "경상북도 경산시", "경상북도 경주시",
    "경상북도 고령군", "경상북도 구미시", "경상북도 김천시", "경상북도 문경시", "경상북도 봉화군", "경상북도 상주시",
    "경상북도 성주군", "경상북도 안동시", "경상북도 영덕군", "경상북도 영양군", "경상북도 영주시", "경상북도 영천시",
    "경상북도 예천군", "경상북도 울릉군", "경상북도 울진군", "경상북도 의성군", "경상북도 청도군", "경상북도 청송군",
    "경상북도 칠곡군", "경상북도 포항시", "광주광역시 광산구", "광주광역시 남구", "광주광역시 동구", "광주광역시 북구",
    "광주광역시 서구", "대구광역시 군위군", "대구광역시 남구", "대구광역시 달서구", "대구광역시 달성군", "대구광역시 동구",
    "대구광역시 북구", "대구광역시 서구", "대구광역시 수성구", "대구광역시 중구", "대전광역시 대덕구", "대전광역시 동구",
    "대전광역시 서구", "대전광역시 유성구", "대전광역시 중구", "부산광역시 강서구", "부산광역시 금정구", "부산광역시 기장군",
    "부산광역시 남구", "부산광역시 동구", "부산광역시 동래구", "부산광역시 북구", "부산광역시 사상구", "부산광역시 사하구",
    "부산광역시 서구", "부산광역시 수영구", "부산광역시 연제구", "부산광역시 영도구", "부산광역시 중구", "부산광역시 부산진구",
    "부산광역시 해운대구", "세종특별자치시", "울산광역시 남구", "울산광역시 동구", "울산광역시 북구", "울산광역시 울주군",
    "울산광역시 중구", "전라남도 강진군", "전라남도 고흥군", "전라남도 곡성군", "전라남도 광양시", "전라남도 구례군",
    "전라남도 나주시", "전라남도 담양군", "전라남도 목포시", "전라남도 무안군", "전라남도 보성군", "전라남도 순천시",
    "전라남도 신안군", "전라남도 여수시", "전라남도 영광군", "전라남도 영암군", "전라남도 완도군", "전라남도 장성군",
    "전라남도 장흥군", "전라남도 진도군", "전라남도 함평군", "전라남도 해남군", "전라남도 화순군", "전북특별자치도 고창군",
    "전북특별자치도 군산시", "전북특별자치도 김제시", "전북특별자치도 남원시", "전북특별자치도 무주군", "전북특별자치도 부안군",
    "전북특별자치도 순창군", "전북특별자치도 완주군", "전북특별자치도 익산시", "전북특별자치도 임실군", "전북특별자치도 장수군",
    "전북특별자치도 전주시", "전북특별자치도 정읍시", "전북특별자치도 진안군", "제주특별자치도 서귀포시", "제주특별자치도 제주시",
    "충청남도 계룡시", "충청남도 공주시", "충청남도 금산군", "충청남도 논산시", "충청남도 당진시", "충청남도 보령시",
    "충청남도 부여군", "충청남도 서산시", "충청남도 서천군", "충청남도 아산시", "충청남도 예산군", "충청남도 천안시",
    "충청남도 청양군", "충청남도 태안군", "충청남도 홍성군", "충청북도 괴산군", "충청북도 단양군", "충청북도 보은군",
    "충청북도 영동군", "충청북도 옥천군", "충청북도 음성군", "충청북도 제천시", "충청북도 증평군", "충청북도 진천군",
    "충청북도 청주시", "충청북도 충주시"
]

    df[region_col] = df[region_col].astype(str).str.strip()
    present_regions = set(df[region_col].unique())
    missing = sorted(set(region_list) - present_regions)

    if missing:
        print(f"📋 누락된 시군구 {len(missing)}개 → 빈 행으로 추가됨:")
        # 누락된 시군구 이름만 있는 빈 DataFrame 생성
        missing_df = pd.DataFrame({region_col: missing})
        df = pd.concat([df, missing_df], ignore_index=True)
    else:
        print("✅ 누락된 시군구 없음")
    # 정렬
    df = df.sort_values(by=region_col).reset_index(drop=True)
    return df


In [258]:
# 데이터 확인
def check_dataframe(
    df: pd.DataFrame,
    expected_rows: int = 229,
    region_col: str = "SGG_NAME",
    save_path: str = "./result.csv"
) -> None:
    """
    DataFrame의 행 개수가 기대값과 일치하면 정렬 후 메세지 출력
    일치하지 않으면 경고 메시지 출력

    Parameters:
        df (pd.DataFrame): 대상 데이터프레임
        expected_rows (int): 기대하는 행 개수 (기본값: 164)
        region_col (str): 정렬 기준 컬럼명 (기본값: 'SGG_NAME')
        save_path (str): 저장 경로 (기본값: './result.csv')
    """
    if len(df) == expected_rows:
        df = df.sort_values(by=region_col).reset_index(drop=True)
        # df.to_csv(save_path, encoding="utf-8-sig", index=False)
        print(f"✅ 행 개수 일치: {save_path} (총 {expected_rows}개 행)")
    else:
        print(f"⚠️ 행 개수 불일치: 현재 {len(df)}개 행 (예상: {expected_rows})")


In [259]:
# 파일 내 첫 번째 컬럼명을 확인하고, 시군구 컬럼(법정동 정보)을 표준화, 정규화하는 함수
def integration_sgg_col(
    filepath: str
)-> pd.DataFrame:
    """
    주어진 CSV 파일의 첫 번째 컬럼명을 확인하여
    시군구 컬럼을 표준화·정규화한 뒤 result.csv로 저장

    Parameters:
        filepath (str): 입력 CSV 파일 경로

    Returns:
        pd.DataFrame: 시군구 컬럼 정제 및 표준화가 완료된 DataFrame
    """
    # 1) 인코딩 오류 방지하며 파일 불러오기
    df = safe_read_csv(filepath)
    first_col = df.columns[0] # 첫 번째 컬럼명

    # 2) 케이스 분기 
    if first_col == "시군구별(1)": # 법정동 컬럼 2개
        df = combine_region_columns(df) # 시군구 컬럼 2개 하나로 합치기
        df = normalize_sido_names(df, column="시군구별", col_cnt=2) # 시도 치환
        df = merge_to_city(df) # 하위 행정구역 통합
        
    elif first_col =="시군구별": # 법정동 컬럼 1개
        df = normalize_sido_names(df, column=first_col, col_cnt=1) # 시도 치환
        df = make_full_sgg_names(df) # 시군구_전체 컬럼 생성
        
    else: # 둘다 아닐때 데이터 초기 전처리 잘못되었으므로 오류 발생 내용 추가
        raise ValueError(
            f"지원하지 않는 첫 컬럼명: '{first_col}'. 허용: '시군구별(1)', '시군구별'"
        )
    
    df = merge_gunwigun(df) # 군위군 데이터 통합
    df = remove_subdistricts(df) # 특례시 하위 행정구역 제거
    df = remove_only_sido_rows(df) # 시도만 있는 행 제거 
    df = check_missing_regions(df, region_col="SGG_NAME")
    check_dataframe(df)

    return df

In [260]:
def data_cleansing_folder(
    folder_path: str, 
    output_folder: str
)-> None:
    """
    모든 하위 폴더의 .csv 파일을 읽어 정제 후 output_folder에 저장

    Parameters:
        folder_path (str): 입력 CSV 파일들이 들어있는 최상위 폴더
        output_folder (str): 정제된 CSV를 저장할 최상위 폴더
    """
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".csv"):  # CSV 파일만 처리
                try:
                    # CSV 읽어서 시군구 컬럼 정제 (사용자 정의 함수)
                    df = integration_sgg_col(os.path.join(root, file))

                    # 원본 폴더 구조 보존: folder_path 기준 상대경로 계산
                    rel = os.path.relpath(root, folder_path)
                    save_dir = os.path.join(output_folder, rel)
                    os.makedirs(save_dir, exist_ok=True)  # 저장할 폴더 생성

                    # 저장 파일명: "_정리.csv" 붙이기
                    save_path = os.path.join(save_dir, file.replace(".csv", "_정리.csv"))

                    # 정제된 CSV 저장
                    df.to_csv(save_path, index=False, encoding="utf-8-sig")
                    print(f"✅ 저장 완료: {save_path}")

                except Exception as e:
                    # 오류 발생 시 로그 출력 (다른 파일 처리는 계속 진행)
                    print(f"❌ 오류 발생 {file}: {e}")

In [261]:
# 입력 폴더
input_path = "../../data/01-1_data-cleansing"
# 츨력 폴더
output_path = "../../data/01-2_data-cleansing"

# 폴더 안 파일들 전체 데이터 정제 실행 함수수
data_cleansing_folder(input_path, output_path)

📋 누락된 시군구 66개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\교원_1인당_학생수_컬럼명변경_정리.csv
📋 누락된 시군구 66개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\대학교 진학률_컬럼명변경_정리.csv
📋 누락된 시군구 132개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\대학교_교원수_컬럼명변경_정리.csv
📋 누락된 시군구 142개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\대학교_수_컬럼명변경_정리.csv
📋 누락된 시군구 66개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\유치원_교원수_컬럼명변경_정리.csv
📋 누락된 시군구 66개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\유치원_수_컬럼명변경_정리.csv
📋 누락된 시군구 66개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\유치원_원아수_컬럼명변경_정리.csv
📋 누락된 시군구 66개 → 빈 행으로 추가됨:
✅ 행 개수 일치: ./result.csv (총 229개 행)
✅ 저장 완료: ../../data/01-2_data-cleansing\교육\인구_천명당_사설학원수

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_2920\1381894033.py:33: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  merged_values = dg_values.combine_first(gw_values)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_2920\4259048582.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[new_col] = df[col_sido].str.strip() + " " + df[col_sigungu].str.strip()
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_2920\4259048582.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 